# Teacher Tests on the Virtual Geometry

Build two virtual-geometry teacher tests for the rollout-finetuned temperature surrogate:

- `case_north_south_wind_existing_source`: same thermal condition as the baseline, but with a north-south relative wind direction simulated by rotating the geometry before rerunning DIGIT velocity;
- `case_top_half_source`: baseline geometry/wind and baseline thermal condition, but the upper half keeps the existing source and the lower half has no source.




In [ ]:
from pathlib import Path
import json
import math
import shutil
import sys
from dataclasses import asdict

import numpy as np
import matplotlib.pyplot as plt
import torch

# Locate the bundle whether the notebook is opened from the repo root or this folder.
cwd = Path.cwd().resolve()
if (cwd / "south_kensington_temperature_3d.py").exists():
    BUNDLE_ROOT = cwd
elif (cwd / "temperature_field_bundle" / "south_kensington_temperature_3d.py").exists():
    BUNDLE_ROOT = cwd / "temperature_field_bundle"
elif (
    cwd / "velocity" / "temperature_field_bundle" / "south_kensington_temperature_3d.py"
).exists():
    BUNDLE_ROOT = cwd / "velocity" / "temperature_field_bundle"
else:
    raise FileNotFoundError(
        "Could not locate temperature_field_bundle from this working directory."
    )

if (
    BUNDLE_ROOT.parent.name == "velocity"
    and (BUNDLE_ROOT.parent.parent / "temperature_field_bundle").exists()
):
    REPO_ROOT = BUNDLE_ROOT.parent.parent
else:
    REPO_ROOT = BUNDLE_ROOT.parent

if str(BUNDLE_ROOT) not in sys.path:
    sys.path.insert(0, str(BUNDLE_ROOT))

from south_kensington_jupyter import SouthKensingtonConfig
from south_kensington_temperature_3d import (
    Temperature3DScenarioConfig,
    run_temperature_3d_pipeline,
    extract_or_load_full_velocity_fields,
    build_surface_temperature_fields,
    solve_temperature_fields_3d,
    save_results,
)
from temperature_surrogate_3d import (
    Temperature3DCase,
    TemperatureSurrogateDataset,
    SurrogateStats,
    UNet3D,
)

ROLLOUT_DIR_CANDIDATES = [
    REPO_ROOT / "temperature_field_bundle" / "rollout_surrogate",
    BUNDLE_ROOT / "rollout_surrogate",
    REPO_ROOT / "Results" / "rollout_surrogate",
    Path.home() / "Velocity" / "temperature_field_bundle" / "rollout_surrogate",
    Path.home() / "Velocity" / "Results" / "rollout_surrogate",
]
ROLLOUT_DIR = next(
    (
        p
        for p in ROLLOUT_DIR_CANDIDATES
        if (p / "test_virtual_geometry_rollout_surrogate.py").exists()
    ),
    None,
)
if ROLLOUT_DIR is None:
    raise FileNotFoundError(
        "Could not locate rollout_surrogate/test_virtual_geometry_rollout_surrogate.py"
    )
if str(ROLLOUT_DIR) not in sys.path:
    sys.path.insert(0, str(ROLLOUT_DIR))
import test_virtual_geometry_rollout_surrogate as rollout

print("Repo root:", REPO_ROOT)
print("Bundle root:", BUNDLE_ROOT)
print("Rollout helper:", ROLLOUT_DIR)

## Run Switches

In [ ]:
RUN_PHYSICAL_CASES = True
RUN_SURROGATE_ROLLOUT = True
FORCE_REGENERATE_PHYSICAL = False

INLET_FLOW = 0.35
REFERENCE_WIND_DIRECTION_DEG = 268.41
NORTH_SOUTH_WIND_ROTATION_DEG = 90
NORTH_SOUTH_REFERENCE_WIND_DIRECTION_DEG = (
    REFERENCE_WIND_DIRECTION_DEG + NORTH_SOUTH_WIND_ROTATION_DEG
) % 360
HISTORY_STEPS = 3
PATCH_SIZE = (16, 32, 32)
OVERLAP = (4, 8, 8)
INFERENCE_MODE = "weighted_tiled"  # Patch-like inference with smooth blending; use 'full_volume' only for comparison.

TEST_ROOT = BUNDLE_ROOT / "outputs" / "teacher_two_case_virtual_geometry_tests"
STATIC_DIR = TEST_ROOT / "static_virtual_complex_v2"
BASE_VELOCITY_CACHE_DIR = TEST_ROOT / "velocity_cache" / "virtual_complex_v2_v035_rot000"
BASE_VELOCITY_DIGIT_DIR = TEST_ROOT / "velocity_digit_outputs" / "virtual_complex_v2_v035_rot000"
NORTH_SOUTH_VELOCITY_CACHE_DIR = (
    TEST_ROOT / "velocity_cache" / "virtual_complex_v2_v035_rot090_recomputed"
)
NORTH_SOUTH_VELOCITY_DIGIT_DIR = (
    TEST_ROOT / "velocity_digit_outputs" / "virtual_complex_v2_v035_rot090_recomputed"
)
CASES_DIR = TEST_ROOT / "cases"
SURROGATE_OUT_DIR = TEST_ROOT / "surrogate_rollout_results"

TEACHER_CASE_IDS = [
    "case_north_south_wind_existing_source",
    "case_top_half_source",
]

# Backward-compatible aliases for helper cells that still expect these names.
VELOCITY_CACHE_DIR = BASE_VELOCITY_CACHE_DIR
VELOCITY_DIGIT_DIR = BASE_VELOCITY_DIGIT_DIR
ALT_VELOCITY_CACHE_DIR = NORTH_SOUTH_VELOCITY_CACHE_DIR
ALT_VELOCITY_DIGIT_DIR = NORTH_SOUTH_VELOCITY_DIGIT_DIR

for path in [
    STATIC_DIR,
    BASE_VELOCITY_CACHE_DIR,
    BASE_VELOCITY_DIGIT_DIR,
    NORTH_SOUTH_VELOCITY_CACHE_DIR,
    NORTH_SOUTH_VELOCITY_DIGIT_DIR,
    CASES_DIR,
    SURROGATE_OUT_DIR,
]:
    path.mkdir(parents=True, exist_ok=True)

print("Test root:", TEST_ROOT)
print("Teacher cases:", TEACHER_CASE_IDS)

## Create the Same Virtual Geometry


In [ ]:
Y_DIM = 112
X_DIM = 112
GRID_RESOLUTION_M = 4.0
HEIGHT_SCALE_M = 4.0
Z_DIM = 10
EMBED_PAD_Y = 8
EMBED_PAD_X = 8

building_mask = np.zeros((Y_DIM, X_DIM), dtype=np.uint8)
building_height = np.zeros((Y_DIM, X_DIM), dtype=np.float32)
land_cover = np.full((Y_DIM, X_DIM), 50, dtype=np.uint8)
yy, xx = np.mgrid[0:Y_DIM, 0:X_DIM]


def add_rotated_rect(center_y, center_x, length, width, angle_deg, height_m):
    angle = np.deg2rad(angle_deg)
    dy = yy - float(center_y)
    dx = xx - float(center_x)
    along = dx * np.cos(angle) + dy * np.sin(angle)
    across = -dx * np.sin(angle) + dy * np.cos(angle)
    rect = (np.abs(along) <= length / 2.0) & (np.abs(across) <= width / 2.0)
    building_mask[rect] = 1
    building_height[rect] = np.maximum(building_height[rect], float(height_m))


# Same complex virtual layout used in the existing virtual-geometry test notebook.
add_rotated_rect(15, 18, 20, 8, -18, 22.0)
add_rotated_rect(18, 44, 28, 7, 11, 34.0)
add_rotated_rect(15, 82, 25, 10, -12, 18.0)
add_rotated_rect(32, 24, 16, 16, 0, 42.0)
add_rotated_rect(36, 62, 18, 12, 28, 30.0)
add_rotated_rect(33, 93, 24, 9, 16, 24.0)
add_rotated_rect(53, 17, 28, 8, 37, 26.0)
add_rotated_rect(55, 45, 14, 26, -7, 46.0)
add_rotated_rect(56, 78, 30, 8, -24, 20.0)
add_rotated_rect(78, 19, 18, 26, -10, 28.0)
add_rotated_rect(82, 50, 30, 12, 18, 38.0)
add_rotated_rect(84, 83, 20, 26, -18, 44.0)
add_rotated_rect(102, 33, 24, 7, -32, 16.0)
add_rotated_rect(101, 68, 28, 8, 9, 26.0)
add_rotated_rect(43, 37, 8, 8, 0, 52.0)
add_rotated_rect(70, 67, 9, 9, 0, 56.0)
add_rotated_rect(25, 71, 7, 7, 0, 48.0)
add_rotated_rect(74, 35, 7, 7, 0, 44.0)

land_cover[7:23, 91:109] = 10
land_cover[88:108, 5:24] = 10
land_cover[46:67, 55:69] = 10
land_cover[21:31, 6:106] = 30
land_cover[66:75, 8:104] = 30
land_cover[:, 51:59] = 30
land_cover[building_mask > 0] = 50

np.save(STATIC_DIR / "building_mask.npy", building_mask)
np.save(STATIC_DIR / "building_height.npy", building_height)
np.save(STATIC_DIR / "land_cover.npy", land_cover)
(STATIC_DIR / "grid.json").write_text(
    json.dumps(
        {
            "resolution_m": GRID_RESOLUTION_M,
            "shape": [Y_DIM, X_DIM],
            "origin_x_m": 0.0,
            "origin_y_m": 0.0,
            "crs": "virtual-local-grid",
        },
        indent=2,
    )
)

print("Building coverage:", round(100 * building_mask.mean(), 1), "%")
print("Max building height:", float(building_height.max()), "m")
print("Land-cover counts:", {int(v): int((land_cover == v).sum()) for v in np.unique(land_cover)})

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
axes[0].imshow(building_height, cmap="magma", origin="upper")
axes[0].set_title("Virtual building height")
axes[1].imshow(land_cover, cmap="tab20", origin="upper")
axes[1].contour(building_mask.astype(float), levels=[0.5], colors="black", linewidths=0.4)
axes[1].set_title("Virtual land cover")
for ax in axes:
    ax.set_xlabel("x index")
    ax.set_ylabel("y index")
plt.show()

## Configure the Two Existing-Pipeline Cases


In [ ]:
baseline_velocity_config = SouthKensingtonConfig(
    static_dir=str(STATIC_DIR),
    raw_buildings_path=str(STATIC_DIR / "no_raw_osm_for_virtual_case.json"),
    geometry_resolution_m=GRID_RESOLUTION_M,
    geometry_rotation_deg=0,
    model_resolution_m=GRID_RESOLUTION_M,
    target_xy=(Y_DIM, X_DIM),
    embed_pad_y=EMBED_PAD_Y,
    embed_pad_x=EMBED_PAD_X,
    z_dim=Z_DIM,
    height_scale_m=HEIGHT_SCALE_M,
    inlet_flow=INLET_FLOW,
    timesteppings=50,
    output_dir=str(BASE_VELOCITY_DIGIT_DIR),
)

north_south_wind_velocity_config = SouthKensingtonConfig(
    static_dir=str(STATIC_DIR),
    raw_buildings_path=str(STATIC_DIR / "no_raw_osm_for_virtual_case.json"),
    geometry_resolution_m=GRID_RESOLUTION_M,
    geometry_rotation_deg=NORTH_SOUTH_WIND_ROTATION_DEG,
    model_resolution_m=GRID_RESOLUTION_M,
    target_xy=(Y_DIM, X_DIM),
    embed_pad_y=EMBED_PAD_Y,
    embed_pad_x=EMBED_PAD_X,
    z_dim=Z_DIM,
    height_scale_m=HEIGHT_SCALE_M,
    inlet_flow=INLET_FLOW,
    timesteppings=50,
    output_dir=str(NORTH_SOUTH_VELOCITY_DIGIT_DIR),
)

north_south_temperature_config = Temperature3DScenarioConfig(
    output_dir=str(CASES_DIR / "case_north_south_wind_existing_source"),
    reuse_velocity_output_dir=str(NORTH_SOUTH_VELOCITY_CACHE_DIR),
    frame_duration_s=90.0,
    diffusion_coeff_m2_s=1.0,
    velocity_scale=1.0,
    ambient_temp_c=24.2,
    inflow_temp_c=24.2,
    ambient_relative_humidity_pct=47.19,
    cloud_cover_fraction=0.47,
    reference_wind_speed_m_s=3.09,
    reference_wind_direction_deg=NORTH_SOUTH_REFERENCE_WIND_DIRECTION_DEG,
    analysis_time_utc="2025-07-25T13:00:00+00:00",
    anthropogenic_heat_flux_w_m2=15.0,
    save_animation=False,
    save_overview_figure=False,
    temperature_solver_backend="auto",
    temperature_solver_device="cuda",
)

top_half_source_temperature_config = Temperature3DScenarioConfig(
    output_dir=str(CASES_DIR / "case_top_half_source"),
    reuse_velocity_output_dir=str(BASE_VELOCITY_CACHE_DIR),
    frame_duration_s=90.0,
    diffusion_coeff_m2_s=1.0,
    velocity_scale=1.0,
    ambient_temp_c=24.2,
    inflow_temp_c=24.2,
    ambient_relative_humidity_pct=47.19,
    cloud_cover_fraction=0.47,
    reference_wind_speed_m_s=3.09,
    reference_wind_direction_deg=REFERENCE_WIND_DIRECTION_DEG,
    analysis_time_utc="2025-07-25T13:00:00+00:00",
    anthropogenic_heat_flux_w_m2=15.0,
    save_animation=False,
    save_overview_figure=False,
    temperature_solver_backend="auto",
    temperature_solver_device="cuda",
)

# Backward-compatible aliases for later helper cells.
velocity_config = baseline_velocity_config
base_temperature_config = top_half_source_temperature_config
alt_wind_velocity_config = north_south_wind_velocity_config

print(
    json.dumps(
        {
            "case_north_south_wind_existing_source": {
                "velocity_config": asdict(north_south_wind_velocity_config),
                "temperature_config": asdict(north_south_temperature_config),
            },
            "case_top_half_source": {
                "velocity_config": asdict(baseline_velocity_config),
                "temperature_config": asdict(top_half_source_temperature_config),
            },
        },
        indent=2,
    )
)

## Source Masks


In [ ]:
def make_source_masks(shape):
    ydim, xdim = shape

    top_half = np.zeros((ydim, xdim), dtype=np.float32)
    top_half[: ydim // 2, :] = 1.0
    return {
        "existing_source": np.ones((ydim, xdim), dtype=np.float32),
        "top_half_source": top_half,
    }


def make_thermal_modifiers(shape):
    ydim, xdim = shape
    return {
        "baseline_thermal": {
            "scale": np.ones((ydim, xdim), dtype=np.float32),
            "offset_c": np.zeros((ydim, xdim), dtype=np.float32),
            "description": "Original baseline thermal boundary fields.",
        },
    }


def apply_boundary_modifiers(boundary_fields, source_mask, thermal_modifier=None):
    out = {k: np.array(v, copy=True) for k, v in boundary_fields.items()}
    mask = source_mask.astype(np.float32)
    if thermal_modifier is None:
        thermal_modifier = make_thermal_modifiers(mask.shape)["baseline_thermal"]
    scale = np.asarray(thermal_modifier["scale"], dtype=np.float32)
    offset_c = np.asarray(thermal_modifier["offset_c"], dtype=np.float32)
    ambient0 = float(out["ambient_temp_series_c"][0])

    ground_base = out["ground_surface_temperature_excess_c"].astype(np.float32)
    building_base = out["building_surface_temperature_excess_c"].astype(np.float32)
    roof_base = out["roof_surface_temperature_excess_3d"].astype(np.float32)

    out["ground_surface_temperature_excess_c"] = (
        ground_base * mask * scale + offset_c * mask
    ).astype(np.float32)
    out["building_surface_temperature_excess_c"] = (
        building_base * mask * scale + offset_c * mask
    ).astype(np.float32)
    out["roof_surface_temperature_excess_3d"] = (
        roof_base * mask[None, :, :] * scale[None, :, :] + offset_c[None, :, :] * mask[None, :, :]
    ).astype(np.float32)

    out["ground_surface_temperature_c"] = (
        ambient0 + out["ground_surface_temperature_excess_c"]
    ).astype(np.float32)
    out["building_surface_temperature_c"] = (
        ambient0 + out["building_surface_temperature_excess_c"]
    ).astype(np.float32)
    if "solid_temperature_3d" in out:
        solid = np.full_like(out["solid_temperature_3d"], ambient0, dtype=np.float32)
        roof_active = np.abs(out["roof_surface_temperature_excess_3d"]) > 1e-6
        solid[roof_active] = ambient0 + out["roof_surface_temperature_excess_3d"][roof_active]
        out["solid_temperature_3d"] = solid
    return out


def apply_source_mask(boundary_fields, source_mask):
    return apply_boundary_modifiers(boundary_fields, source_mask, None)

## Generate Physical Ground-Truth Cases


In [ ]:
def physical_case_complete(case_dir):
    required = [
        case_dir / "temperature_fields_3d_c.npy",
        case_dir / "temperature_fields_3d_masked_c.npy",
        case_dir / "temperature_3d_run_summary.json",
    ]
    return all(path.exists() and path.stat().st_size > 0 for path in required)


def physical_case_matches_spec(case_dir, spec):
    if not physical_case_complete(case_dir):
        return False
    metadata_path = case_dir / "source_test_metadata.json"
    if not metadata_path.exists():
        return False
    metadata = json.loads(metadata_path.read_text())
    velocity_meta = metadata.get("velocity_config", {})
    temperature_meta = metadata.get("temperature_config", {})
    if int(velocity_meta.get("geometry_rotation_deg", -1)) != int(
        spec["velocity_config"].geometry_rotation_deg
    ):
        return False
    if str(temperature_meta.get("reuse_velocity_output_dir")) != str(
        spec["temperature_config"].reuse_velocity_output_dir
    ):
        return False
    if not np.isclose(
        float(temperature_meta.get("ambient_temp_c", np.nan)),
        float(spec["temperature_config"].ambient_temp_c),
    ):
        return False
    if not np.isclose(
        float(temperature_meta.get("reference_wind_direction_deg", np.nan)),
        float(spec["temperature_config"].reference_wind_direction_deg),
    ):
        return False
    if metadata.get("source_mask_name") != spec.get("source_mask_name"):
        return False
    if metadata.get("thermal_modifier_name", "baseline_thermal") != spec.get(
        "thermal_modifier_name", "baseline_thermal"
    ):
        return False
    if not np.isclose(
        float(temperature_meta.get("anthropogenic_heat_flux_w_m2", np.nan)),
        float(spec["temperature_config"].anthropogenic_heat_flux_w_m2),
    ):
        return False
    return True


test_specs = [
    {
        "case_id": "case_north_south_wind_existing_source",
        "velocity_config": north_south_wind_velocity_config,
        "temperature_config": north_south_temperature_config,
        "source_mask_name": "existing_source",
        "thermal_modifier_name": "baseline_thermal",
        "display_rotation_deg": -NORTH_SOUTH_WIND_ROTATION_DEG,
        "note": "North-south relative wind direction via 90-degree geometry rotation; DIGIT velocity is recomputed for the rotated geometry.",
    },
    {
        "case_id": "case_top_half_source",
        "velocity_config": baseline_velocity_config,
        "temperature_config": top_half_source_temperature_config,
        "source_mask_name": "top_half_source",
        "thermal_modifier_name": "baseline_thermal",
        "display_rotation_deg": 0,
        "note": "Baseline wind/geometry/thermal condition; upper half keeps the existing source and lower half has no source.",
    },
]
test_spec_by_case_id = {spec["case_id"]: spec for spec in test_specs}


def expected_case_dirs(require_matching=True):
    dirs = []
    for case_id in TEACHER_CASE_IDS:
        case_dir = CASES_DIR / case_id
        spec = test_spec_by_case_id[case_id]
        if require_matching:
            if physical_case_matches_spec(case_dir, spec):
                dirs.append(case_dir)
        elif physical_case_complete(case_dir):
            dirs.append(case_dir)
    return dirs


def print_physical_case_status():
    print("Physical case status:")
    missing = []
    for case_id in TEACHER_CASE_IDS:
        case_dir = CASES_DIR / case_id
        spec = test_spec_by_case_id[case_id]
        complete = physical_case_complete(case_dir)
        matches = physical_case_matches_spec(case_dir, spec)
        status = "ready" if matches else ("stale" if complete else "missing")
        print(f"  {case_id}: {status} -> {case_dir}")
        if not matches:
            missing.append(case_id)
    return missing


if RUN_PHYSICAL_CASES:
    manifest = []
    for spec in test_specs:
        case_dir = Path(spec["temperature_config"].output_dir)
        if physical_case_matches_spec(case_dir, spec) and not FORCE_REGENERATE_PHYSICAL:
            print("[skip]", spec["case_id"])
            summary = json.loads((case_dir / "temperature_3d_run_summary.json").read_text())
        else:
            if physical_case_complete(case_dir) and not FORCE_REGENERATE_PHYSICAL:
                print("[rerun stale]", spec["case_id"], flush=True)
            else:
                print("[run]", spec["case_id"], flush=True)
            velocity_results, fields, cache_reused = extract_or_load_full_velocity_fields(
                spec["velocity_config"],
                spec["temperature_config"],
            )
            base_boundary_fields = build_surface_temperature_fields(
                fields,
                spec["velocity_config"],
                spec["temperature_config"],
            )
            masks = make_source_masks(fields["study_area_mask_2d"].shape)
            thermal_modifiers = make_thermal_modifiers(fields["study_area_mask_2d"].shape)
            source_mask = masks[spec["source_mask_name"]]
            thermal_modifier = thermal_modifiers.get(
                spec.get("thermal_modifier_name", "baseline_thermal"),
                thermal_modifiers["baseline_thermal"],
            )
            boundary = apply_boundary_modifiers(base_boundary_fields, source_mask, thermal_modifier)
            temperature_results = solve_temperature_fields_3d(
                fields,
                boundary,
                spec["velocity_config"],
                spec["temperature_config"],
            )
            summary = save_results(
                velocity_results,
                fields,
                boundary,
                temperature_results,
                spec["velocity_config"],
                spec["temperature_config"],
                cache_reused,
            )
            np.save(case_dir / "source_mask.npy", source_mask.astype(np.float32))
            metadata = {
                "case_id": spec["case_id"],
                "case_dir": str(case_dir),
                "note": spec["note"],
                "source_mask_name": spec["source_mask_name"],
                "source_active_fraction": float(
                    np.mean(source_mask[fields["study_area_mask_2d"]] > 0)
                ),
                "thermal_modifier_name": spec.get("thermal_modifier_name", "baseline_thermal"),
                "thermal_modifier_description": thermal_modifier.get("description", ""),
                "thermal_modifier_scale_min": float(np.nanmin(thermal_modifier["scale"])),
                "thermal_modifier_scale_max": float(np.nanmax(thermal_modifier["scale"])),
                "thermal_modifier_offset_min_c": float(np.nanmin(thermal_modifier["offset_c"])),
                "thermal_modifier_offset_max_c": float(np.nanmax(thermal_modifier["offset_c"])),
                "display_rotation_deg": spec["display_rotation_deg"],
                "velocity_config": asdict(spec["velocity_config"]),
                "temperature_config": asdict(spec["temperature_config"]),
            }
            (case_dir / "source_test_metadata.json").write_text(json.dumps(metadata, indent=2))
        manifest.append(
            {
                "case_id": spec["case_id"],
                "case_dir": str(case_dir),
                "note": spec["note"],
                "summary_path": str(case_dir / "temperature_3d_run_summary.json"),
                "source_mask_name": spec["source_mask_name"],
                "thermal_modifier_name": spec.get("thermal_modifier_name", "baseline_thermal"),
                "display_rotation_deg": spec["display_rotation_deg"],
            }
        )

    (TEST_ROOT / "source_test_manifest.json").write_text(json.dumps(manifest, indent=2))
    print("Saved manifest:", TEST_ROOT / "source_test_manifest.json")
else:
    print(
        "RUN_PHYSICAL_CASES is False. Set it to True to generate the physical ground-truth cases."
    )

missing_or_stale_cases = print_physical_case_status()
if RUN_PHYSICAL_CASES and missing_or_stale_cases:
    raise RuntimeError(
        "Physical generation did not produce all requested cases: "
        + ", ".join(missing_or_stale_cases)
    )

## Load Rollout-Finetuned Surrogate


In [ ]:
checkpoint_candidates = [
    rollout.ROLLOUT_CHECKPOINT,
    REPO_ROOT
    / "Results"
    / "rollout_surrogate"
    / "outputs"
    / "best_temperature_3d_unet_surrogate_rollout.pt",
    REPO_ROOT
    / "Results"
    / "rollout_surrogate"
    / "outputs"
    / "last_temperature_3d_unet_surrogate_rollout.pt",
    rollout.FALLBACK_CHECKPOINT,
    REPO_ROOT
    / "Results"
    / "surrogate_modelling"
    / "output"
    / "temperature"
    / "best_temperature_3d_unet_surrogate.pt",
]
checkpoint_path = next((p for p in checkpoint_candidates if p.exists()), None)
if checkpoint_path is None:
    raise FileNotFoundError(
        "No surrogate checkpoint found in candidates:\n"
        + "\n".join(str(p) for p in checkpoint_candidates)
    )

checkpoint = torch.load(checkpoint_path, map_location="cpu")
config = checkpoint.get("config", {})
# Keep this test on the original patch-wise inference path: predict overlapping patches,
# then blend/stitch them back into the full domain.
INFERENCE_MODE = "weighted_tiled"
rollout.CORRECTION_ITERATIONS = int(
    config.get("correction_iterations", rollout.CORRECTION_ITERATIONS)
)
stats = SurrogateStats(**checkpoint["stats"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = rollout.load_model_from_checkpoint(checkpoint, device)
checkpoint_kind = (
    "rollout-finetuned surrogate"
    if "rollout" in str(checkpoint_path).lower()
    else "one-step/raw surrogate"
)
checkpoint_summary = {
    "device": str(device),
    "checkpoint_path": str(checkpoint_path),
    "checkpoint_kind": checkpoint_kind,
    "inference_mode": INFERENCE_MODE,
    "correction_iterations": int(rollout.CORRECTION_ITERATIONS),
    "epoch": checkpoint.get("epoch"),
    "best_val_loss": checkpoint.get("best_val_loss", checkpoint.get("val_loss")),
    "config": {
        k: config.get(k)
        for k in ["base_channels", "history_steps", "patch_size", "norm", "correction_iterations"]
        if k in config
    },
    "stats": {
        "temp_mean": float(stats.temp_mean),
        "temp_std": float(stats.temp_std),
    },
}
print("Surrogate checkpoint summary:")
print(json.dumps(checkpoint_summary, indent=2))

## Run Surrogate Rollout on Each Generated Physical Case


In [ ]:
@torch.no_grad()
def predict_volume_full(model, x_full, stats, device):
    xt = torch.from_numpy(x_full[None, ...]).to(device, non_blocking=True)
    with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
        pred_norm = model(xt)[0, 0]
    return (pred_norm.float().cpu().numpy() * stats.temp_std + stats.temp_mean).astype(np.float32)


def autoregressive_rollout_full_volume(case, model, stats, device):
    case = rollout.apply_z_crop(case)
    helper = TemperatureSurrogateDataset(
        [case],
        stats,
        history_steps=rollout.HISTORY_STEPS,
        patch_size=None,
        samples_per_case=1,
        random_crop=False,
        deterministic_sampling=True,
    )
    crop = (slice(None), slice(None), slice(None))
    valid_mask = helper._fluid_mask(case, crop).astype(bool)
    history = [
        np.asarray(frame, dtype=np.float32) for frame in case.temperature[: rollout.HISTORY_STEPS]
    ]
    pred_frames = []
    true_frames = []
    frame_ids = []

    for target_frame in range(rollout.HISTORY_STEPS, case.temperature.shape[0]):
        t = target_frame - 1
        print(f"[full-volume rollout] predicting frame {target_frame}", flush=True)
        correction_history = history[-rollout.HISTORY_STEPS :]
        future_guess = rollout.initial_future_guess(correction_history)
        pred = None
        for iteration in range(rollout.CORRECTION_ITERATIONS):
            informed_guess = rollout.impose_temperature_constraints(
                future_guess,
                case,
                stats,
                t,
                correction_history,
                valid_mask,
            )
            x_full = rollout.make_input(
                case, helper, stats, correction_history, informed_guess, t, device
            )
            pred = predict_volume_full(model, x_full, stats, device)
            pred = rollout.impose_temperature_constraints(
                pred, case, stats, t, correction_history, valid_mask
            )
            alpha = float(iteration + 1) / float(rollout.CORRECTION_ITERATIONS)
            future_guess = informed_guess * (1.0 - alpha) + pred * alpha
            future_guess = rollout.impose_temperature_constraints(
                future_guess,
                case,
                stats,
                t,
                correction_history,
                valid_mask,
            )
            pred = future_guess.astype(np.float32)
        pred_frames.append(pred.astype(np.float32))
        true_frames.append(np.asarray(case.temperature[target_frame], dtype=np.float32))
        frame_ids.append(target_frame)
        history.append(pred.astype(np.float32))

    return (
        np.asarray(pred_frames, dtype=np.float32),
        np.asarray(true_frames, dtype=np.float32),
        valid_mask,
        np.asarray(frame_ids, dtype=np.int32),
    )


def evaluate_rollout_case(case_dir):
    case = rollout.apply_z_crop(Temperature3DCase(case_dir))
    if INFERENCE_MODE == "full_volume":
        pred_seq, true_seq, valid_mask, frame_ids = autoregressive_rollout_full_volume(
            case, model, stats, device
        )
    elif INFERENCE_MODE in {"weighted_tiled", "tiled"}:
        pred_seq, true_seq, valid_mask, frame_ids = rollout.autoregressive_rollout(
            case, model, stats, device, inference_mode=INFERENCE_MODE
        )
    else:
        raise ValueError("INFERENCE_MODE must be 'full_volume', 'weighted_tiled', or 'tiled'.")
    err_seq = pred_seq - true_seq
    frame_bias_c = [float(np.mean(frame_err[valid_mask])) for frame_err in err_seq]
    frame_mae_c = [float(np.mean(np.abs(frame_err[valid_mask]))) for frame_err in err_seq]
    frame_rmse_c = [float(np.sqrt(np.mean(frame_err[valid_mask] ** 2))) for frame_err in err_seq]
    metrics = {
        "case_id": case_dir.name,
        "case_dir": str(case_dir),
        "checkpoint": str(checkpoint_path),
        "mode": "autoregressive_rollout",
        "inference_mode": INFERENCE_MODE,
        "frames": [int(v) for v in frame_ids],
        "first_predicted_frame": int(frame_ids[0]),
        "final_frame": int(frame_ids[-1]),
        "first_step_bias_c": float(np.mean(err_seq[0][valid_mask])),
        "first_step_mae_c": float(np.mean(np.abs(err_seq[0][valid_mask]))),
        "first_step_rmse_c": float(np.sqrt(np.mean(err_seq[0][valid_mask] ** 2))),
        "final_bias_c": float(np.mean(err_seq[-1][valid_mask])),
        "final_mae_c": float(np.mean(np.abs(err_seq[-1][valid_mask]))),
        "final_rmse_c": float(np.sqrt(np.mean(err_seq[-1][valid_mask] ** 2))),
        "rollout_bias_c": float(np.mean(err_seq[:, valid_mask])),
        "rollout_mae_c": float(np.mean(np.abs(err_seq[:, valid_mask]))),
        "rollout_rmse_c": float(np.sqrt(np.mean(err_seq[:, valid_mask] ** 2))),
        "frame_bias_c": frame_bias_c,
        "frame_mae_c": frame_mae_c,
        "frame_rmse_c": frame_rmse_c,
    }
    out_dir = SURROGATE_OUT_DIR / case_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / "rollout_metrics.json").write_text(json.dumps(metrics, indent=2))
    np.save(out_dir / "pred_temperature_c.npy", pred_seq.astype(np.float32))
    np.save(out_dir / "true_temperature_c.npy", true_seq.astype(np.float32))
    np.save(out_dir / "frame_ids.npy", frame_ids.astype(np.int32))
    return metrics, pred_seq, true_seq, valid_mask, frame_ids


if RUN_SURROGATE_ROLLOUT:
    case_dirs = expected_case_dirs(require_matching=True)
    if not case_dirs:
        print(
            "No physical cases found. Set RUN_PHYSICAL_CASES=True and run the generation cell first."
        )
    all_metrics = []
    last_payload = None
    for idx, case_dir in enumerate(case_dirs, start=1):
        print(
            f"[surrogate {idx}/{len(case_dirs)}] {case_dir.name} | inference={INFERENCE_MODE}",
            flush=True,
        )
        metrics, pred_seq, true_seq, valid_mask, frame_ids = evaluate_rollout_case(case_dir)
        all_metrics.append(metrics)
        last_payload = (case_dir, pred_seq, true_seq, valid_mask, frame_ids)
    if all_metrics:
        metrics_path = SURROGATE_OUT_DIR / "all_source_distribution_rollout_metrics.json"
        metrics_path.write_text(json.dumps(all_metrics, indent=2))
        print("Saved metrics:", metrics_path)
        print(
            json.dumps(
                [
                    {
                        k: m[k]
                        for k in [
                            "case_id",
                            "inference_mode",
                            "rollout_mae_c",
                            "rollout_rmse_c",
                            "final_mae_c",
                        ]
                    }
                    for m in all_metrics
                ],
                indent=2,
            )
        )
else:
    print("RUN_SURROGATE_ROLLOUT is False.")

## Compare Metrics Across Source Tests


In [ ]:
metrics_path = SURROGATE_OUT_DIR / "all_source_distribution_rollout_metrics.json"
if metrics_path.exists():
    all_metrics = [
        m for m in json.loads(metrics_path.read_text()) if m.get("case_id") in TEACHER_CASE_IDS
    ]
    if not all_metrics:
        print(
            "Metric summary exists, but none of the requested teacher cases are present. Run the rollout cell first."
        )
    labels = [
        m["case_id"].replace("case_", "").replace("_existing_source", "").replace("_", " ")
        for m in all_metrics
    ]
    if all_metrics:
        mae = np.array([m["rollout_mae_c"] for m in all_metrics], dtype=np.float32)
        rmse = np.array([m["rollout_rmse_c"] for m in all_metrics], dtype=np.float32)
        final_mae = np.array([m["final_mae_c"] for m in all_metrics], dtype=np.float32)

        x = np.arange(len(labels))
        width = 0.28
        fig, ax = plt.subplots(figsize=(max(9, len(labels) * 1.2), 4.8), constrained_layout=True)
        ax.bar(x - width, mae, width, label="rollout MAE")
        ax.bar(x, rmse, width, label="rollout RMSE")
        ax.bar(x + width, final_mae, width, label="final-frame MAE")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right")
        ax.set_ylabel("deg C")
        ax.set_title("Surrogate error across teacher tests")
        ax.legend()
        plt.show()
else:
    print("No metric summary found yet.")

## Visualise Rollout Frame 48 at z=6


In [ ]:
VIS_CASE_IDS = TEACHER_CASE_IDS
VIS_FRAME = 48
VIS_Z_LEVEL = 6


def display_quarter_turns(rotation_deg):
    rotation_deg = int(round(float(rotation_deg))) % 360
    if rotation_deg % 90 != 0:
        raise ValueError("Display rotation must be a multiple of 90 degrees.")
    return (rotation_deg // 90) % 4


def rotate_display_2d(arr, turns):
    return np.rot90(arr, k=turns).copy() if turns else arr


def masked_display_slice(volume, mask, z_idx, turns):
    slice_mask = mask[z_idx]
    out = np.asarray(volume[z_idx], dtype=np.float32).copy()
    out[~slice_mask] = np.nan
    return rotate_display_2d(out, turns)


def case_display_label(case_id):
    return case_id.replace("case_", "").replace("_existing_source", "").replace("_", " ")


plot_rows = []
for VIS_CASE_ID in VIS_CASE_IDS:
    print(f"[rollout visualize] {VIS_CASE_ID} | frame={VIS_FRAME} | z={VIS_Z_LEVEL}")
    vis_case_dir = CASES_DIR / VIS_CASE_ID
    vis_out_dir = SURROGATE_OUT_DIR / VIS_CASE_ID
    spec = test_spec_by_case_id[VIS_CASE_ID]
    if not physical_case_matches_spec(vis_case_dir, spec):
        print(
            f"Stale or missing physical case for {VIS_CASE_ID}. Run the physical generation cell first."
        )
        continue

    pred_path = vis_out_dir / "pred_temperature_c.npy"
    true_path = vis_out_dir / "true_temperature_c.npy"
    frame_ids_path = vis_out_dir / "frame_ids.npy"
    if not (pred_path.exists() and true_path.exists() and frame_ids_path.exists()):
        print(f"Missing rollout outputs for {VIS_CASE_ID}. Run the rollout cell first.")
        continue

    pred_seq = np.load(pred_path)
    true_seq = np.load(true_path)
    frame_ids = np.load(frame_ids_path)
    matches = np.where(frame_ids == VIS_FRAME)[0]
    if matches.size == 0:
        print(
            f"Frame {VIS_FRAME} not found for {VIS_CASE_ID}; available range is {int(frame_ids[0])}-{int(frame_ids[-1])}."
        )
        continue
    frame_idx = int(matches[0])

    case = Temperature3DCase(vis_case_dir)
    helper = TemperatureSurrogateDataset(
        [case],
        stats,
        history_steps=HISTORY_STEPS,
        patch_size=None,
        samples_per_case=1,
        random_crop=False,
    )
    valid_mask = helper._fluid_mask(case, (slice(None), slice(None), slice(None))).astype(bool)
    building_mask_2d = np.load(case._velocity_cache_dir() / "building_mask_2d.npy").astype(bool)

    z_idx = min(VIS_Z_LEVEL, pred_seq.shape[1] - 1)
    err = pred_seq[frame_idx] - true_seq[frame_idx]
    display_turns = display_quarter_turns(spec.get("display_rotation_deg", 0))
    display_building_mask = rotate_display_2d(building_mask_2d, display_turns)
    slice_err = err[z_idx][valid_mask[z_idx]]

    plot_rows.append(
        {
            "case_id": VIS_CASE_ID,
            "label": case_display_label(VIS_CASE_ID),
            "frame": int(frame_ids[frame_idx]),
            "z_idx": int(z_idx),
            "true": masked_display_slice(true_seq[frame_idx], valid_mask, z_idx, display_turns),
            "pred": masked_display_slice(pred_seq[frame_idx], valid_mask, z_idx, display_turns),
            "err": masked_display_slice(err, valid_mask, z_idx, display_turns),
            "building_mask": display_building_mask,
            "bias": float(np.mean(slice_err)),
            "mae": float(np.mean(np.abs(slice_err))),
            "display_rotation_deg": int(spec.get("display_rotation_deg", 0)),
        }
    )

if not plot_rows:
    print("No rollout rows available to plot.")
else:
    vmin = 22.0
    vmax = 32.0
    all_err_values = np.concatenate(
        [np.abs(row["err"][np.isfinite(row["err"])]).ravel() for row in plot_rows]
    )
    err_lim = max(float(np.percentile(all_err_values, 98)), 1e-6)

    temp_cmap = plt.get_cmap("YlOrRd").copy()
    temp_cmap.set_bad("white")
    err_cmap = plt.get_cmap("coolwarm").copy()
    err_cmap.set_bad("white")

    fig, axes = plt.subplots(
        len(plot_rows), 3, figsize=(15, 4.4 * len(plot_rows)), constrained_layout=True
    )
    axes = np.atleast_2d(axes)
    for row_idx, row in enumerate(plot_rows):
        im0 = axes[row_idx, 0].imshow(
            row["true"], cmap=temp_cmap, vmin=vmin, vmax=vmax, origin="upper"
        )
        axes[row_idx, 0].set_title(
            f"{row['label']}\nPhysical model T | frame {row['frame']} | z={row['z_idx']}"
        )
        axes[row_idx, 1].imshow(row["pred"], cmap=temp_cmap, vmin=vmin, vmax=vmax, origin="upper")
        axes[row_idx, 1].set_title(
            f"{row['label']}\nRollout-finetuned surrogate T | z={row['z_idx']}"
        )
        im2 = axes[row_idx, 2].imshow(
            row["err"], cmap=err_cmap, vmin=-err_lim, vmax=err_lim, origin="upper"
        )
        axes[row_idx, 2].set_title(
            f"Error: surrogate - physical\nbias={row['bias']:+.3f} C | MAE={row['mae']:.3f} C"
        )
        for ax in axes[row_idx]:
            ax.contour(
                row["building_mask"].astype(float), levels=[0.5], colors="black", linewidths=0.4
            )
            ax.set_xlabel("x index")
            ax.set_ylabel("y index")

    fig.colorbar(
        im0, ax=axes[:, :2], shrink=0.85, fraction=0.025, pad=0.02, label="Temperature (deg C)"
    )
    fig.colorbar(im2, ax=axes[:, 2], shrink=0.85, fraction=0.025, pad=0.02, label="deg C")
    fig.suptitle(
        f"Rollout-finetuned surrogate | frame {VIS_FRAME} | z={VIS_Z_LEVEL} | checkpoint: {Path(checkpoint_path).name}",
        fontsize=14,
    )
    plt.show()

## Visualise One-Step Checkpoint Frame 48 at z=6


In [ ]:
ONE_STEP_FRAME = 48
ONE_STEP_Z_LEVEL = 6
ONE_STEP_INFERENCE_MODE = "weighted_tiled"
ONE_STEP_CORRECTION_ITERATIONS = 1

one_step_checkpoint_candidates = [
    BUNDLE_ROOT
    / "outputs"
    / "temperature_3d_unet_surrogate"
    / "best_temperature_3d_unet_surrogate.pt",
    REPO_ROOT
    / "temperature_field_bundle"
    / "outputs"
    / "temperature_3d_unet_surrogate"
    / "best_temperature_3d_unet_surrogate.pt",
    REPO_ROOT
    / "Results"
    / "surrogate_modelling"
    / "output"
    / "temperature"
    / "best_temperature_3d_unet_surrogate.pt",
    rollout.FALLBACK_CHECKPOINT,
]
one_step_checkpoint_path = next((p for p in one_step_checkpoint_candidates if p.exists()), None)
if one_step_checkpoint_path is None:
    raise FileNotFoundError(
        "No one-step surrogate checkpoint found in candidates:\n"
        + "\n".join(str(p) for p in one_step_checkpoint_candidates)
    )

one_step_checkpoint = torch.load(one_step_checkpoint_path, map_location="cpu")
one_step_config = one_step_checkpoint.get("config", {})
one_step_stats = SurrogateStats(**one_step_checkpoint["stats"])
one_step_model = rollout.load_model_from_checkpoint(one_step_checkpoint, device)
one_step_summary = {
    "device": str(device),
    "checkpoint_path": str(one_step_checkpoint_path),
    "checkpoint_kind": "one-step/raw surrogate",
    "inference_mode": ONE_STEP_INFERENCE_MODE,
    "correction_iterations": ONE_STEP_CORRECTION_ITERATIONS,
    "target_frame": ONE_STEP_FRAME,
    "input_history_frames": list(range(ONE_STEP_FRAME - HISTORY_STEPS, ONE_STEP_FRAME)),
    "epoch": one_step_checkpoint.get("epoch"),
    "best_val_loss": one_step_checkpoint.get("best_val_loss", one_step_checkpoint.get("val_loss")),
    "config": {
        k: one_step_config.get(k)
        for k in ["base_channels", "history_steps", "patch_size", "norm"]
        if k in one_step_config
    },
    "stats": {
        "temp_mean": float(one_step_stats.temp_mean),
        "temp_std": float(one_step_stats.temp_std),
    },
}
print("One-step surrogate checkpoint summary:")
print(json.dumps(one_step_summary, indent=2))


@torch.no_grad()
def predict_one_step_frame(case_dir, target_frame=ONE_STEP_FRAME):
    case = rollout.apply_z_crop(Temperature3DCase(case_dir))
    helper = TemperatureSurrogateDataset(
        [case],
        one_step_stats,
        history_steps=HISTORY_STEPS,
        patch_size=None,
        samples_per_case=1,
        random_crop=False,
        deterministic_sampling=True,
    )
    valid_mask = helper._fluid_mask(case, (slice(None), slice(None), slice(None))).astype(bool)
    if target_frame < HISTORY_STEPS or target_frame >= case.temperature.shape[0]:
        raise ValueError(
            f"target_frame must be in [{HISTORY_STEPS}, {case.temperature.shape[0] - 1}], got {target_frame}."
        )

    t = int(target_frame) - 1
    history_c = [
        np.asarray(case.temperature[target_frame - HISTORY_STEPS + offset], dtype=np.float32)
        for offset in range(HISTORY_STEPS)
    ]
    future_guess = rollout.initial_future_guess(history_c)
    future_guess = rollout.impose_temperature_constraints(
        future_guess, case, one_step_stats, t, history_c, valid_mask
    )

    pred = None
    for iteration in range(ONE_STEP_CORRECTION_ITERATIONS):
        x_full = rollout.make_input(
            case, helper, one_step_stats, history_c, future_guess, t, device
        )
        pred = rollout.predict_volume(
            one_step_model, x_full, one_step_stats, device, inference_mode=ONE_STEP_INFERENCE_MODE
        )
        pred = rollout.impose_temperature_constraints(
            pred, case, one_step_stats, t, history_c, valid_mask
        )
        future_guess = pred.astype(np.float32)
    if pred is None:
        raise RuntimeError("ONE_STEP_CORRECTION_ITERATIONS must be at least 1.")
    return (
        pred.astype(np.float32),
        np.asarray(case.temperature[target_frame], dtype=np.float32),
        valid_mask,
        case,
    )


plot_rows = []
for VIS_CASE_ID in VIS_CASE_IDS:
    print(
        f"[one-step visualize] {VIS_CASE_ID} | input frames {ONE_STEP_FRAME - HISTORY_STEPS}-{ONE_STEP_FRAME - 1} -> frame {ONE_STEP_FRAME} | z={ONE_STEP_Z_LEVEL}"
    )
    vis_case_dir = CASES_DIR / VIS_CASE_ID
    spec = test_spec_by_case_id[VIS_CASE_ID]
    if not physical_case_matches_spec(vis_case_dir, spec):
        print(
            f"Stale or missing physical case for {VIS_CASE_ID}. Run the physical generation cell first."
        )
        continue

    pred, true, valid_mask, case = predict_one_step_frame(vis_case_dir, target_frame=ONE_STEP_FRAME)
    building_mask_2d = np.load(case._velocity_cache_dir() / "building_mask_2d.npy").astype(bool)
    z_idx = min(ONE_STEP_Z_LEVEL, pred.shape[0] - 1)
    err = pred - true
    display_turns = display_quarter_turns(spec.get("display_rotation_deg", 0))
    display_building_mask = rotate_display_2d(building_mask_2d, display_turns)
    slice_err = err[z_idx][valid_mask[z_idx]]

    plot_rows.append(
        {
            "case_id": VIS_CASE_ID,
            "label": case_display_label(VIS_CASE_ID),
            "frame": int(ONE_STEP_FRAME),
            "z_idx": int(z_idx),
            "true": masked_display_slice(true, valid_mask, z_idx, display_turns),
            "pred": masked_display_slice(pred, valid_mask, z_idx, display_turns),
            "err": masked_display_slice(err, valid_mask, z_idx, display_turns),
            "building_mask": display_building_mask,
            "bias": float(np.mean(slice_err)),
            "mae": float(np.mean(np.abs(slice_err))),
        }
    )

if not plot_rows:
    print("No one-step rows available to plot.")
else:
    vmin = 22.0
    vmax = 32.0
    all_err_values = np.concatenate(
        [np.abs(row["err"][np.isfinite(row["err"])]).ravel() for row in plot_rows]
    )
    err_lim = max(float(np.percentile(all_err_values, 98)), 1e-6)

    temp_cmap = plt.get_cmap("YlOrRd").copy()
    temp_cmap.set_bad("white")
    err_cmap = plt.get_cmap("coolwarm").copy()
    err_cmap.set_bad("white")

    fig, axes = plt.subplots(
        len(plot_rows), 3, figsize=(15, 4.4 * len(plot_rows)), constrained_layout=True
    )
    axes = np.atleast_2d(axes)
    for row_idx, row in enumerate(plot_rows):
        im0 = axes[row_idx, 0].imshow(
            row["true"], cmap=temp_cmap, vmin=vmin, vmax=vmax, origin="upper"
        )
        axes[row_idx, 0].set_title(
            f"{row['label']}\nPhysical model T | frame {row['frame']} | z={row['z_idx']}"
        )
        axes[row_idx, 1].imshow(row["pred"], cmap=temp_cmap, vmin=vmin, vmax=vmax, origin="upper")
        axes[row_idx, 1].set_title(f"{row['label']}\nOne-step surrogate T | z={row['z_idx']}")
        im2 = axes[row_idx, 2].imshow(
            row["err"], cmap=err_cmap, vmin=-err_lim, vmax=err_lim, origin="upper"
        )
        axes[row_idx, 2].set_title(
            f"Error: surrogate - physical\nbias={row['bias']:+.3f} C | MAE={row['mae']:.3f} C"
        )
        for ax in axes[row_idx]:
            ax.contour(
                row["building_mask"].astype(float), levels=[0.5], colors="black", linewidths=0.4
            )
            ax.set_xlabel("x index")
            ax.set_ylabel("y index")

    fig.colorbar(
        im0, ax=axes[:, :2], shrink=0.85, fraction=0.025, pad=0.02, label="Temperature (deg C)"
    )
    fig.colorbar(im2, ax=axes[:, 2], shrink=0.85, fraction=0.025, pad=0.02, label="deg C")
    fig.suptitle(
        f"One-step/raw surrogate | input frames {ONE_STEP_FRAME - HISTORY_STEPS}-{ONE_STEP_FRAME - 1} -> frame {ONE_STEP_FRAME} | z={ONE_STEP_Z_LEVEL} | checkpoint: {Path(one_step_checkpoint_path).name}",
        fontsize=14,
    )
    plt.show()